# Lesson 02 - Meneroka Rangka Kerja Ejen Microsoft

**Rangka Kerja Ejen Microsoft (MAF)** ialah rangka kerja bersatu untuk membina ejen AI. Ia menyediakan seni bina yang bersih dan boleh disusun dengan empat blok binaan teras:

- **Client** – menyambung ke titik akhir model AI dan mengendalikan komunikasi
- **Agent** – membungkus klien dengan arahan dan definisi alat
- **Tools** – meluaskan keupayaan ejen dengan fungsi tersuai yang boleh dipanggil oleh model
- **Session** – mengekalkan sejarah perbualan untuk interaksi berbilang giliran

Dalam pelajaran ini, kita akan membina **ejen tempahan perjalanan** yang memeriksa ketersediaan destinasi menggunakan konsep-konsep ini.


## Persediaan


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Memahami Seni Bina Rangka Kerja Ejen

Rangka Kerja Ejen Microsoft mengikuti seni bina berlapis:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – Sebuah `AzureAIProjectAgentProvider` yang menyambung ke penyebaran Azure OpenAI. Ia mengendalikan pengesahan, format permintaan, dan pemahaman maklum balas.
2. **Agent** – Dibuat dari klien melalui `provider.create_agent()`, ejen menggabungkan akses model dengan arahan (petunjuk sistem) dan alat.
3. **Tools** – Fungsi Python yang dihiasi dengan `@tool` yang boleh dipanggil oleh ejen untuk melakukan tindakan atau mendapatkan data.
4. **Session** – Objek `AgentSession` (dibuat melalui `agent.create_session()`) yang menyimpan sejarah perbualan, membolehkan dialog berbilang pusingan di mana ejen mengingati konteks sebelum ini.

Mari bina setiap lapisan langkah demi langkah.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Menambah Alat dengan Hiasan @tool

Alat membenarkan agen mengambil tindakan selain daripada menjana teks. Hiasan `@tool` menukar fungsi Python biasa menjadi sesuatu yang boleh dipanggil oleh agen.

Perkara utama:
- Gunakan `Annotated[type, "description"]` supaya model memahami setiap parameter.
- Docstring menjadi penerangan alat yang dilihat oleh model.
- `approval_mode="never_require"` bermaksud alat dijalankan secara automatik tanpa pengesahan pengguna.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Mewujudkan Agen dengan Alat

Sekarang kita menggabungkan klien, arahan, dan alat menjadi satu agen. `instructions` bertindak sebagai arahan sistem — ia menentukan persona dan tingkah laku agen tersebut.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Perbualan Pelbagai Giliran dengan Sesi

`AgentSession` (dibuat melalui `agent.create_session()`) menyimpan jejak semua mesej dalam satu perbualan. Dengan menghantar sesi yang sama kepada setiap panggilan `agent.run()`, ejen mempunyai akses kepada sejarah penuh perbualan dan boleh merujuk kembali kepada mesej-mesej terdahulu.

Kami menghantar `tools=[check_destination_availability]` supaya ejen boleh menggunakan pemeriksa ketersediaan kami semasa setiap giliran.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Summary

Dalam pelajaran ini anda menjelajahi empat tiang Rangka Kerja Agen Microsoft:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` menyambung ke Azure OpenAI dengan pengesahan berasaskan kelayakan |
| **Agent** | `provider.create_agent()` menggabungkan sambungan model dengan arahan dan nama |
| **Tools** | Penghias `@tool` mendedahkan fungsi Python untuk agen panggil |
| **Session** | `agent.create_session()` mengekalkan sejarah perbualan merentasi beberapa giliran |

Blok binaan ini digabungkan untuk mencipta agen yang boleh mengadakan perbualan semula jadi, memanggil fungsi luaran, dan mengekalkan konteks — asas untuk corak agen lebih maju dalam pelajaran seterusnya.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:  
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk memastikan ketepatan, sila maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
